# Taller Integrado: Regresión Lineal Bayesiana - Caso Uber Bogotá

Este notebook contiene el flujo completo para analizar el experimento de Uber sobre los tiempos de espera y su impacto en la demanda de viajes compartidos (`trips_pool`), utilizando Inferencia Bayesiana con **PyMC**.

**Facultad de Ciencia de Datos - Universidad Externado de Colombia**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

## 1. Carga y Limpieza de Datos

Primero cargamos la base `data/BaseUBER.xlsx` y realizamos las conversiones necesarias para que las variables sean numéricas.

In [ ]:
df = pd.read_excel('data/BaseUBER.xlsx')

# 1. Convertir wait_time a número (extraer el dígito)
df['wait_time_num'] = df['wait_time'].astype(str).str.extract('(\d+)').astype(float)

# 2. Convertir booleanos a enteros
df['commute_num'] = df['commute'].astype(int)
df['treat_num'] = df['treat'].astype(int)

print("Primeros registros:")
display(df.head())

print("\nResumen estadístico de la variable objetivo:")
print(df['trips_pool'].describe())

## 2. Análisis Exploratorio de Datos (EDA)

Revisamos cómo se distribuyen los viajes según el tiempo de espera y el horario.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x='wait_time_num', y='trips_pool', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Viajes según Tiempo de Espera')

sns.boxplot(x='commute_num', y='trips_pool', data=df, ax=ax[1], palette='muted')
ax[1].set_title('Viajes según Horario (Pico vs Normal)')

plt.show()

## 3. Matriz de Correlación

Evaluamos la relación entre todas las variables numéricas para identificar redundancias.

In [ ]:
numeric_cols = ['wait_time_num', 'treat_num', 'commute_num', 'trips_pool', 
                'trips_express', 'rider_cancellations', 'total_driver_payout', 'total_matches']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True)
plt.title("Matriz de Correlación de Pearson")
plt.show()

## 4. Modelo Lineal Bayesiano

Planteamos el modelo:
$$y \sim N(\beta_0 + \beta_{wait} X_{wait} + \beta_{commute} X_{commute}, \sigma)$$

Usaremos priors débilmente informativos:
- $\beta \sim N(0, 10)$
- $\sigma \sim HalfNormal(300)$

In [ ]:
with pm.Model() as model_uber:
    # Priors
    beta_0 = pm.Normal('beta_0', mu=1400, sigma=500)
    beta_wait = pm.Normal('beta_wait', mu=0, sigma=10)
    beta_commute = pm.Normal('beta_commute', mu=0, sigma=10)
    sigma = pm.HalfNormal('sigma', sigma=300)
    
    # Verosimilitud (Linear model)
    mu = beta_0 + beta_wait * df['wait_time_num'] + beta_commute * df['commute_num']
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=df['trips_pool'])
    
    # Muestreo
    print("Iniciando el muestreo MCMC (esto puede tardar unos segundos)...")
    trace = pm.sample(draws=2000, tune=1000, chains=4, random_seed=42)
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

## 5. Diagnósticos y Resultados

In [ ]:
# 1. Resumen de parámetros
display(az.summary(trace, round_to=2))

# 2. Traceplot (Convergencia)
az.plot_trace(trace)
plt.tight_layout()
plt.show()

# 3. Forest Plot (Intervalos de Credibilidad HDI 95%)
az.plot_forest(trace, var_names=['beta_wait', 'beta_commute'], combined=True)
plt.axvline(0, color='red', linestyle='--')
plt.title("Intervalos HDI al 95%: Efecto de las Variables")
plt.show()

# 4. PPC (Chequeo Predictivo a Posteriori)
az.style.use("arviz-darkgrid")
az.plot_ppc(ppc)
plt.show()

### Conclusiones Principales:
1. **Efecto Nulo**: Si los intervalos de `beta_wait` o `beta_commute` cruzan el cero (lo cual es muy probable en este conjunto de datos), concluimos que no hay evidencia estadística suficiente para decir que cambiar de 2 a 5 minutos afecta la demanda.
2. **Inelasticidad**: Para Uber en mercados como Bogotá, esto es ideal: pueden aumentar la eficiencia logística (esperar más tiempo para enlazar rutas) sin perder clientes.